***Preparação***

In [34]:
import numpy as np
import time

def generate_data(num_samples=500):
    """Gera dados com a regra condicional de temperatura."""
    X, y = [], []
    for _ in range(num_samples):
        temp, pressure, vibration = np.random.rand(), np.random.rand(), np.random.rand()
        label = 0 # Reprovado por padrão
        
        # A regra de exceção: temperatura na zona de perigo -> reprovação imediata
        if 0.7 <= temp <= 0.9:
            label = 0
        # A regra normal: só checada se a primeira for falsa
        elif pressure > 0.6 and vibration < 0.4:
            label = 1
            
        X.append([temp, pressure, vibration])
        y.append([label])
    return np.array(X), np.array(y)

    

***Modelo de rede neural multi perceptron tradicional***

In [35]:
class SmartMLP:
    """
    Uma Rede Neural Multilayer Perceptron (MLP) tradicional.
    A arquitetura é 3 (entrada) -> 5 (oculta 1) -> 4 (oculta 2) -> 1 (saída).
    Esta rede não possui nenhuma lógica de comporta customizada.
    """
    def __init__(self, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Os pesos e vieses são armazenados como matrizes e vetores NumPy.
        # Inicializamos com valores pequenos e aleatórios para quebrar a simetria.
        self.weights_h1 = np.random.randn(input_size, h1_size) * 0.1
        self.bias_h1 = np.zeros(h1_size)
        
        # A camada H2 é uma camada densa padrão, assim como a H1.
        self.weights_h2 = np.random.randn(h1_size, h2_size) * 0.1
        self.bias_h2 = np.zeros(h2_size)
        
        self.weights_out = np.random.randn(h2_size, output_size) * 0.1
        self.bias_out = np.zeros(output_size)
        
        print("Standard MLP (3-5-4-1) criada.")

    def _sigmoid(self, x):
        """Função de ativação sigmoide. Coloca os valores entre 0 e 1."""
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        """Derivada da sigmoide, necessária para o backpropagation."""
        return x * (1 - x)

    def predict(self, inputs):
        """
        Realiza o 'Forward Pass': calcula a predição da rede para uma dada entrada.
        """
        # Da entrada para a camada oculta 1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        
        # Da camada oculta 1 para a camada oculta 2
        self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
        
        # Da camada oculta 2 para a camada de saída
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        
        return final_output

    def train(self, X, y, epochs=2000, learning_rate=0.1):
        """
        Executa o treinamento da rede usando o algoritmo de backpropagation.
        """
        print(f"Iniciando treinamento por {epochs} épocas...")
        start_time = time.time()
        
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- 1. Forward Pass ---
                # A predição é calculada para obter o erro.
                final_output = self.predict(inputs)

                # --- 2. Backward Pass (Cálculo dos Gradientes) ---
                # O erro é a diferença entre o esperado и o obtido.
                error = expected - final_output
                total_error += np.sum(error**2)

                # Calcula o gradiente (delta) para cada camada, de trás para frente.
                d_output = error * self._sigmoid_derivative(final_output)
                
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                error_h1 = d_h2.dot(self.weights_h2.T)
                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- 3. Atualização dos Pesos e Vieses ---
                # Ajusta os parâmetros da rede na direção que minimiza o erro.
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                self.bias_h2 += d_h2 * learning_rate
                
                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate
            
            if (epoch + 1) % 500 == 0:
                print(f"  Época {epoch + 1}/{epochs}, Erro Total: {total_error:.4f}")

        end_time = time.time()
        print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.\n")


# --- PASSO 3: Script Principal de Execução ---

# Gerar o conjunto de dados
X_train, y_train = generate_data()

# Instanciar a rede neural tradicional
standard_mlp = SmartMLP()

# Treinar a rede
standard_mlp.train(X_train, y_train)

# --- PASSO 4: Testar o Desempenho em Casos Específicos ---

print("="*40)
print("      AVALIAÇÃO DO MODELO TRADICIONAL")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = standard_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.4f} -> {resultado_final}")

Standard MLP (3-5-4-1) criada.
Iniciando treinamento por 2000 épocas...
  Época 500/2000, Erro Total: 22.6590
  Época 1000/2000, Erro Total: 19.6685
  Época 1500/2000, Erro Total: 18.5965
  Época 2000/2000, Erro Total: 18.0486
Treinamento concluído em 31.16 segundos.

      AVALIAÇÃO DO MODELO TRADICIONAL

--- Caso de Teste: Aprovação Normal ---
Entrada: [0.2 0.8 0.1], Resultado Esperado: 1
  Predição da Standard MLP: 0.9976 -> Aprovado

--- Caso de Teste: Reprovação Normal ---
Entrada: [0.3 0.2 0.2], Resultado Esperado: 0
  Predição da Standard MLP: 0.0073 -> Reprovado

--- Caso de Teste: Reprovação por Temperatura (Crítico) ---
Entrada: [0.8 0.9 0.1], Resultado Esperado: 0
  Predição da Standard MLP: 0.0506 -> Reprovado


***Modelo de rede neural multi perceptron com comporta fraca***


In [33]:
class TrainableGatedPerceptron:
    """Neurônio com comporta suave para ser usado na camada H2."""
    def __init__(self, num_inputs):
        self.weights = np.random.randn(num_inputs) * 0.1
        self.bias = np.random.randn() * 0.1
        
        # Parâmetros fixos da comporta para este exemplo
        self.control_input_index = 0 # Será controlado pelo 1º neurônio da H1
        self.inhibit_range = (0.7, 0.9)
        self.sharpness = 50 # Quão "súbita" é a transição da comporta

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _soft_gate_function(self, control_signal):
        A, B = self.inhibit_range
        bump = self._sigmoid(self.sharpness * (control_signal - A)) * \
               self._sigmoid(-self.sharpness * (control_signal - B))
        return 1 - bump

    def forward(self, inputs):
        self.last_inputs = inputs
        control_signal = inputs[self.control_input_index]
        self.last_gate_value = self._soft_gate_function(control_signal)
        
        z = np.dot(self.weights, inputs) + self.bias
        self.last_activation = self._sigmoid(z)
        
        return float(self.last_activation * self.last_gate_value)

class GatedMLP:
    """Nossa MLP com a camada H2 'gated' e um loop de treinamento simplificado."""
    def __init__(self, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Armazenamento dos pesos e vieses
        self.weights_h1 = np.random.randn(input_size, h1_size) * 0.1
        self.bias_h1 = np.zeros(h1_size)
        
        # A camada H2 é uma lista de neurônios customizados
        self.hidden_layer_2 = [TrainableGatedPerceptron(h1_size) for _ in range(h2_size)]
        
        self.weights_out = np.random.randn(h2_size, output_size) * 0.1
        self.bias_out = np.zeros(output_size)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        return x * (1 - x)

    def predict(self, inputs):
        """Forward pass para predição."""
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        self.h2_output = np.array([float(neuron.forward(self.h1_output)) for neuron in self.hidden_layer_2], dtype=float)
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        return final_output

    def train(self, X, y, epochs=2000, learning_rate=0.1):
        """Loop de treinamento simplificado."""
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- Forward Pass ---
                final_output = self.predict(inputs)

                # --- Backward Pass (Backpropagation Simplificado) ---
                error = expected - final_output
                total_error += error**2

                # Gradiente da camada de saída
                d_output = error * self._sigmoid_derivative(final_output)

                # Gradiente da camada H2
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                # Gradiente da camada H1
                # Aqui, propagamos o erro através dos pesos de cada neurônio H2
                error_h1 = np.zeros(self.weights_h1.shape[1])
                for i, neuron in enumerate(self.hidden_layer_2):
                    error_h1 += d_h2[i] * neuron.weights * neuron.last_gate_value

                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- Atualização dos Pesos e Vieses ---
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                for i, neuron in enumerate(self.hidden_layer_2):
                    neuron.weights += neuron.last_inputs * d_h2[i] * learning_rate
                    neuron.bias += d_h2[i] * learning_rate

                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate

            if (epoch + 1) % 500 == 0:
                print(f"Época {epoch + 1}/{epochs}, Erro: {total_error[0]:.4f}")


# --- Passo 3: Criar, Treinar e Testar a Rede ---

# Gerar dados
X_train, y_train = generate_data()

# Criar a instância da rede
mlp = GatedMLP()

# Treinar a rede
print("--- Iniciando Treinamento ---")
mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

# Testar com casos específicos
print("--- Testando a Rede Treinada ---\n")

# Caso 1: Condições ideais, deve ser APROVADO (1)
test_1 = np.array([0.2, 0.8, 0.1]) # temp baixa, pressão alta, vibração baixa
pred_1 = mlp.predict(test_1)
print(f"Teste 1 (Aprovação Esperada): {test_1}")
print(f"Predição: {pred_1[0]:.4f} -> {'Aprovado' if pred_1 > 0.5 else 'Reprovado'}")
print(f"  Valor da comporta (H2[0]): {mlp.hidden_layer_2[0].last_gate_value:.4f}")
print("-" * 20)

# Caso 2: Condições ruins (pressão baixa), deve ser REPROVADO (0)
test_2 = np.array([0.3, 0.2, 0.2]) # temp baixa, pressão baixa, vibração baixa
pred_2 = mlp.predict(test_2)
print(f"Teste 2 (Reprovação Esperada): {test_2}")
print(f"Predição: {pred_2[0]:.4f} -> {'Aprovado' if pred_2 > 0.5 else 'Reprovado'}")
print(f"  Valor da comporta (H2[0]): {mlp.hidden_layer_2[0].last_gate_value:.4f}")
print("-" * 20)

# Caso 3: CONDIÇÃO DA COMPORTA! Temp alta, mas outras condições são boas. Deve ser REPROVADO (0)
test_3 = np.array([0.8, 0.9, 0.1]) # temp na zona de perigo!
pred_3 = mlp.predict(test_3)
print(f"Teste 3 (Reprovação por Temperatura Esperada): {test_3}")
print(f"Predição: {pred_3[0]:.4f} -> {'Aprovado' if pred_3 > 0.5 else 'Reprovado'}")
print(f"  Valor da comporta (H2[0]): {mlp.hidden_layer_2[0].last_gate_value:.4f}")
print("-" * 20)

--- Iniciando Treinamento ---
Época 500/2000, Erro: 18.0337
Época 1000/2000, Erro: 17.1988
Época 1500/2000, Erro: 13.3653
Época 2000/2000, Erro: 10.9462
--- Treinamento Concluído ---

--- Testando a Rede Treinada ---

Teste 1 (Aprovação Esperada): [0.2 0.8 0.1]
Predição: 0.9936 -> Aprovado
  Valor da comporta (H2[0]): 1.0000
--------------------
Teste 2 (Reprovação Esperada): [0.3 0.2 0.2]
Predição: 0.0000 -> Reprovado
  Valor da comporta (H2[0]): 1.0000
--------------------
Teste 3 (Reprovação por Temperatura Esperada): [0.8 0.9 0.1]
Predição: 0.7888 -> Aprovado
  Valor da comporta (H2[0]): 1.0000
--------------------


***Modelo de rede neural multi perceptron com comporta forte***

In [ ]:
class SmartMLP:
    """
    Uma Rede Neural Multilayer Perceptron (MLP) tradicional.
    A arquitetura é 3 (entrada) -> 5 (oculta 1) -> 4 (oculta 2) -> 1 (saída).
    Esta rede não possui nenhuma lógica de comporta customizada.
    """
    def __init__(self, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Os pesos e vieses são armazenados como matrizes e vetores NumPy.
        # Inicializamos com valores pequenos e aleatórios para quebrar a simetria.
        self.weights_h1 = np.random.randn(input_size, h1_size) * 0.1
        self.bias_h1 = np.zeros(h1_size)
        
        # A camada H2 é uma camada densa padrão, assim como a H1.
        self.weights_h2 = np.random.randn(h1_size, h2_size) * 0.1
        self.bias_h2 = np.zeros(h2_size)
        
        self.weights_out = np.random.randn(h2_size, output_size) * 0.1
        self.bias_out = np.zeros(output_size)
        
        print("Standard MLP (3-5-4-1) criada.")

    def _sigmoid(self, x):
        """Função de ativação sigmoide. Coloca os valores entre 0 e 1."""
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        """Derivada da sigmoide, necessária para o backpropagation."""
        return x * (1 - x)

    def predict(self, inputs):
        """
        Realiza o 'Forward Pass': calcula a predição da rede para uma dada entrada.
        """
        # Da entrada para a camada oculta 1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        
        # Da camada oculta 1 para a camada oculta 2
        self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
        
        # Da camada oculta 2 para a camada de saída
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        
        return final_output

    def train(self, X, y, epochs=2000, learning_rate=0.1):
        """
        Executa o treinamento da rede usando o algoritmo de backpropagation.
        """
        print(f"Iniciando treinamento por {epochs} épocas...")
        start_time = time.time()
        
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- 1. Forward Pass ---
                # A predição é calculada para obter o erro.
                final_output = self.predict(inputs)

                # --- 2. Backward Pass (Cálculo dos Gradientes) ---
                # O erro é a diferença entre o esperado и o obtido.
                error = expected - final_output
                total_error += np.sum(error**2)

                # Calcula o gradiente (delta) para cada camada, de trás para frente.
                d_output = error * self._sigmoid_derivative(final_output)
                
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                error_h1 = d_h2.dot(self.weights_h2.T)
                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- 3. Atualização dos Pesos e Vieses ---
                # Ajusta os parâmetros da rede na direção que minimiza o erro.
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                self.bias_h2 += d_h2 * learning_rate
                
                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate
            
            if (epoch + 1) % 500 == 0:
                print(f"  Época {epoch + 1}/{epochs}, Erro Total: {total_error:.4f}")

        end_time = time.time()
        print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.\n")



# --- PASSO 3: Script Principal de Execução ---

# Gerar o conjunto de dados
X_train, y_train = generate_data()

# Instanciar a rede neural tradicional
standard_mlp = SmartMLP()

# Treinar a rede
standard_mlp.train(X_train, y_train)

# --- PASSO 4: Testar o Desempenho em Casos Específicos ---

print("="*40)
print("      AVALIAÇÃO DO MODELO TRADICIONAL")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = standard_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.4f} -> {resultado_final}")

Standard MLP (3-5-4-1) criada.
Iniciando treinamento por 2000 épocas...
  Época 500/2000, Erro Total: 24.1225
  Época 1000/2000, Erro Total: 22.0462
  Época 1500/2000, Erro Total: 11.2723
  Época 2000/2000, Erro Total: 10.4928
Treinamento concluído em 33.26 segundos.

      AVALIAÇÃO DO MODELO TRADICIONAL

--- Caso de Teste: Aprovação Normal ---
Entrada: [0.2 0.8 0.1], Resultado Esperado: 1
  Predição da Standard MLP: 0.9571 -> Aprovado

--- Caso de Teste: Reprovação Normal ---
Entrada: [0.3 0.2 0.2], Resultado Esperado: 0
  Predição da Standard MLP: 0.0000 -> Reprovado

--- Caso de Teste: Reprovação por Temperatura (Crítico) ---
Entrada: [0.8 0.9 0.1], Resultado Esperado: 0
  Predição da Standard MLP: 0.8352 -> Aprovado
